In [ ]:
library(reticulate)
library(Seurat)
library(sceasy)
library(Matrix)

In [ ]:
st <-readRDS('data/spatial/processed_data/spatial.rds')
st

In [ ]:
output_dir <-'data/test'
counts <- LayerData(st, assay = "Spatial", layer = "counts")
writeMM(t(counts), file.path(output_dir, "matrix.mtx"))

# 2. Metadata
write.csv(st@meta.data, file.path(output_dir, "obs.csv"))

write.csv(data.frame(gene = rownames(st)), file.path(output_dir, "var.csv"))

spatial_all <- data.frame()
for(img in names(st@images)) {
  coords <- st@images[[img]]@coordinates
  coords$image_id <- img
  coords$barcode <- rownames(coords)
  spatial_all <- rbind(spatial_all, coords)
}
write.csv(spatial_all, file.path(output_dir, "spatial.csv"), row.names = FALSE)

cat("✅ Exported! Run Python script now.\n")

In [ ]:
import scanpy as sc
import pandas as pd
import scipy.io as sio
import decoupler as dc
import pyreadr
import numpy as np

In [ ]:
data_dir = "data/test"

print("📖 Loading data...")
X = sio.mmread(f"{data_dir}/matrix.mtx").tocsr()
obs = pd.read_csv(f"{data_dir}/obs.csv", index_col=0)
var = pd.read_csv(f"{data_dir}/var.csv", index_col='gene')
spatial = pd.read_csv(f"{data_dir}/spatial.csv")

print("🔨 Building AnnData...")
adata = sc.AnnData(X=X, obs=obs, var=var)

spatial = spatial.set_index('barcode').loc[adata.obs_names]
adata.obsm['spatial'] = spatial[['imagerow', 'imagecol']].values
adata.obsm['spatial_array'] = spatial[['row', 'col']].values
adata.obs['image_id'] = spatial['image_id'].values
adata.obs['tissue'] = spatial['tissue'].values
adata.obs['array_row'] = spatial['row'].values
adata.obs['array_col'] = spatial['col'].values

adata.uns['spatial'] = {}
for img_id in adata.obs['image_id'].unique():
    adata.uns['spatial'][img_id] = {
        'scalefactors': {
            'spot_diameter_fullres': 89.43,
            'tissue_hires_scalef': 0.08
        }
    }

print("\n✅ AnnData created:")
print(adata)
print(f"\nCoordinates: {list(adata.obsm.keys())}")
print(f"Images: {adata.obs['image_id'].value_counts().to_dict()}")

adata.write(f"{data_dir}/tonsil_spatial.h5ad")
print(f"\n💾 Saved to: {data_dir}/tonsil_spatial.h5ad")

In [ ]:
adata = sc.read_h5ad('data/test/tonsil_spatial.h5ad')
adata

In [ ]:
adata = adata[adata.obs['donor_id']=='BCLL-8-T']
adata

In [ ]:
df = pd.read_csv("data/test/dot.csv", index_col=0)
df

In [ ]:
adata.obsm["proportion"] = df.values
adata.uns["cell_type"] = df.columns.tolist()

In [ ]:
cell_types = adata.uns["cell_type"]
idx = cell_types.index('epithelial')
prop = adata.obsm["proportion"][:, idx]
#weighted_X = adata.X * prop[:, np.newaxis]
#prop[:, np.newaxis]

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["norm"] = adata.X.copy()

In [ ]:
adata.obs

In [ ]:
sc.pl.embedding(
    adata,
    basis="spatial_rot",
    size=100,
)

In [ ]:
dc.pp.knn(adata, key="spatial", bw=100, cutoff=0.1)
adata.obs["conn"] = adata.obsp["spatial_connectivities"][0].toarray().ravel()
sc.pl.embedding(adata, basis="spatial_array", color="conn", size=100)

In [ ]:
genes = [
    'ICAM1',
    'HLA-DPA1',
    'HLA-DRB1',
    'HLA-DPB1',
    'CD74',
    'LTB',
    'CD81',
    'JCHAIN',
    'MMP9'
]
#adata.X = adata.obsp["spatial_connectivities"].dot(adata.X.multiply(prop[:, np.newaxis]))
adata.X = adata.obsp["spatial_connectivities"].dot(adata.X)
#sc.pl.embedding(adata, basis="spatial_rot",color=genes, size=100)
sc.pl.embedding(adata, basis="spatial_rot",color=genes, size=100)

In [ ]:
collectri = dc.op.collectri(organism="human")
dc.mt.ulm(data=adata, net=collectri)

In [ ]:
score = dc.pp.get_obsm(adata=adata, key="score_ulm")
score

In [ ]:
score.var_names

In [ ]:
tf='AHRR'
sc.pl.embedding(score, basis="spatial_rot", color=[tf], cmap="RdBu_r", vcenter=0, size=100, title=[f"{tf} score", "niches"])
sc.pl.violin(score, keys=tf, rotation=90, ylabel=f"{tf} score")